In [ ]:
from dataclasses import dataclass
from pathlib import Path

import kornia.augmentation as K
import matplotlib.pyplot as plt
import numpy as np
import onnx
import onnxruntime as ort
import random
import torch
import torch.nn as nn
from huggingface_hub import HfApi
from torchinfo import summary
from torchvision import datasets, transforms
from tqdm import tqdm
import wandb


# Set all seeds for full reproducibility
def set_seed(seed: int = 42):
    """Set seeds for all sources of randomness"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
DTYPE = torch.float32 if torch.cuda.is_available() else torch.bfloat16 if torch.backends.mps.is_available() else torch.float32
print(f"Device: {DEVICE}")

In [ ]:
@dataclass
class Config:
    image_size: int = 32
    num_classes: int = 10
    val_split: float = 0.1
    batch_size: int = 512
    epochs: int = 10
    lr: float = 1e-3
    weight_decay: float = 1e-4
    dropout: float = 0.1
    onnx_path: Path = Path("../data/mnist_cnn.onnx")
    opset_version: int = 17


cfg = Config()
print(f"Config: {cfg}")

In [ ]:

base_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Resize(cfg.image_size),
    ]
)

cache_path = Path("../data/qmnist_cache.pt")

if cache_path.exists():
    print(f"Loading cached tensors from {cache_path}")
    cache = torch.load(cache_path)
    all_images, all_labels = cache["images"], cache["labels"]
else:
    print("Building cache from raw dataset...")
    raw_train = datasets.QMNIST(root="../data/", train=True, download=True, transform=base_transform)
    raw_test = datasets.QMNIST(root="../data/", train=False, download=True, transform=base_transform)

    images_list, labels_list = [], []
    for img, label in tqdm(torch.utils.data.ConcatDataset([raw_train, raw_test]), desc="Caching"):
        images_list.append(img)
        labels_list.append(label)

    all_images = torch.stack(images_list)   # (N, 1, H, W)
    all_labels = torch.tensor(labels_list)  # (N,)
    torch.save({"images": all_images, "labels": all_labels}, cache_path)
    print(f"Cached {len(all_images)} samples to {cache_path}")

total_size = len(all_images)

# Split indices: 80 / 10 / 10
gen = torch.Generator().manual_seed(42)
perm = torch.randperm(total_size, generator=gen)

train_size = int(0.8 * total_size)
val_size   = int(0.1 * total_size)

train_idx = perm[:train_size]
val_idx   = perm[train_size:train_size + val_size]
test_idx  = perm[train_size + val_size:]


class CachedDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]


train_split = CachedDataset(all_images[train_idx], all_labels[train_idx])
val_split   = CachedDataset(all_images[val_idx],   all_labels[val_idx])
test_split  = CachedDataset(all_images[test_idx],  all_labels[test_idx])

train_loader = torch.utils.data.DataLoader(train_split, batch_size=cfg.batch_size, shuffle=True,  num_workers=4, pin_memory=False)
val_loader   = torch.utils.data.DataLoader(val_split,   batch_size=cfg.batch_size, shuffle=False, num_workers=4, pin_memory=False)
test_loader  = torch.utils.data.DataLoader(test_split,  batch_size=cfg.batch_size, shuffle=False, num_workers=4, pin_memory=False)

# GPU augmentation pipeline (applied in training loop after .to(DEVICE))
gpu_aug = nn.Sequential(
    K.RandomRotation(degrees=45),
    K.RandomResizedCrop((cfg.image_size, cfg.image_size), scale=(0.75, 1.2), ratio=(0.9, 1.1)),
).to(DEVICE)

print(f"Train size: {len(train_split)} ({len(train_split) / total_size:.1%})")
print(f"Val size:   {len(val_split)} ({len(val_split) / total_size:.1%})")
print(f"Test size:  {len(test_split)} ({len(test_split) / total_size:.1%})")


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()

x, y = next(iter(test_loader))
for i in range(10):
    img = x[i].squeeze().numpy()
    label = y[i].item()
    axes[i].imshow(img, cmap="gray")
    axes[i].set_title(f"Label: {label}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
class Attention(nn.Module):
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads

        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        self.out_proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, C, H, W = x.shape
        qkv = self.qkv(x.flatten(2).transpose(1, 2)).reshape(
            B, H * W, 3, self.num_heads, self.head_dim
        ).permute(2, 0, 3, 1, 4)

        q, k, v = qkv[0], qkv[1], qkv[2]
        attn_weights = (q @ k.transpose(-2, -1)) / (self.head_dim**0.5)
        attn_weights = attn_weights.softmax(dim=-1)

        attn_output = (attn_weights @ v).transpose(1, 2).reshape(B, H * W, -1)
        return self.out_proj(attn_output).transpose(1, 2).reshape(B, C, H, W)

class LayerNorm2d(nn.Module):
    def __init__(self, num_features, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(num_features))
        self.bias = nn.Parameter(torch.zeros(num_features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(dim=[2, 3], keepdim=True)
        var = x.var(dim=[2, 3], keepdim=True, unbiased=False)
        x_normalized = (x - mean) / torch.sqrt(var + self.eps)
        return self.weight.view(1, -1, 1, 1) * x_normalized + self.bias.view(
            1, -1, 1, 1
        )

class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, m: int = 2):
        super().__init__()
        self.dw = nn.Conv2d(
            in_channels,
            in_channels,
            kernel_size=7,
            padding=3,
            groups=in_channels,
            bias=False,
        )

        self.conv1 = nn.Conv2d(in_channels, out_channels * m, kernel_size=1, bias=False)
        self.conv2 = nn.Conv2d(
            out_channels * m, out_channels, kernel_size=1, bias=False
        )

        self.act = nn.GELU()
        self.norm = LayerNorm2d(in_channels)

        self.skip = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Conv2d(in_channels, out_channels, kernel_size=1)
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.norm(self.dw(x))
        x = self.conv2(self.act(self.conv1(x)))
        return x + residual

class Model(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.block1 = ConvBlock(1, 8)
        self.block2 = ConvBlock(8, 16)
        self.block3 = ConvBlock(16, 32)
        self.block4 = ConvBlock(32, 16)
        self.classifier = nn.Linear(16 * 2 * 2, num_classes)
        self.down = nn.AvgPool2d(2)
        # self.attn = Attention(32)

    def normalize_inputs(self, x):
        # normalize each image to mean 0, and std 1
        mean = x.mean(dim=[2, 3], keepdim=True)
        std = x.std(dim=[2, 3], keepdim=True) + 1e-6
        return (x - mean) / std

    def forward(self, x):
        x = self.normalize_inputs(x)
        x = self.down(self.block1(x))
        x = self.down(self.block2(x))
        x = self.down(self.block3(x))
        # x = self.attn(x)
        x = self.down(self.block4(x))
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


model = Model(num_classes=cfg.num_classes).to(DEVICE, dtype=DTYPE)
summary(
    model,
    input_size=(1, 1, cfg.image_size, cfg.image_size),
    device=DEVICE,
    depth=1,
    dtypes=[DTYPE],
)

In [ ]:
@torch.inference_mode()
def evaluate(loader, criterion, run, split="val"):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images, labels = images.to(DEVICE, dtype=DTYPE), labels.to(DEVICE)
        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * labels.size(0)
        total_correct += (outputs.argmax(dim=1) == labels).sum().item()
        total_samples += labels.size(0)

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    run.log({f"{split}/{split}_loss": avg_loss, f"{split}/{split}_accuracy": accuracy})
    return avg_loss, accuracy

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=4e-4)

In [ ]:
run = wandb.init(project="mnist-cnn", config=cfg.__dict__)
run.watch(model, log="all")

In [ ]:
for epoch in range(cfg.epochs):
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{cfg.epochs}")
    for images, labels in pbar:
        images, labels = images.to(DEVICE, dtype=DTYPE), labels.to(DEVICE)
        images = gpu_aug(images)
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # clip norms
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)

        pbar.set_postfix_str(
            f"loss: {loss.item():.4f}, acc: {(outputs.argmax(dim=1) == labels).float().mean().item():.4f}, lr: {scheduler.get_last_lr()[0]:.2e}, grad_norm: {grad_norm:.4f}"
        )
        run.log(
            {
                "train/train_loss": loss.item(),
                "train/train_accuracy": (outputs.argmax(dim=1) == labels)
                .float()
                .mean()
                .item(),
                "train/lr": scheduler.get_last_lr()[0],
                "train/grad_norm": grad_norm.item(),
            }
        )

    val_loss, val_acc = evaluate(val_loader, criterion, run)
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    scheduler.step()

In [ ]:
test_loss, test_acc = evaluate(test_loader, criterion, run, split="test")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
# show model's gradient flow
def print_grad_flow(model, csv=False, weight=True, bias=True):
    if csv:
        print(
            "name,grad_mean,grad_std,grad_max,grad_min,grad_norm,weight_norm,ratio,num_zeros"
        )
        for name, param in model.named_parameters():
            if param.grad is not None:
                if not weight and "weight" in name or not bias and "bias" in name:
                    continue
                grad_mean = param.grad.mean()
                grad_std = param.grad.std()
                grad_max = param.grad.max()
                grad_min = param.grad.min()
                grad_norm = param.grad.norm()
                weight_norm = param.norm()
                ratio = grad_norm / (weight_norm + 1e-6)
                num_zeros = (param.grad == 0).sum().item()

                print(
                    f"{name},{grad_mean:.4f},{grad_std:.4f},{grad_max:.4f},{grad_min:.4f},{grad_norm:.4f},{weight_norm:.4f},{ratio:.4f},{num_zeros}"
                )
    else:
        print(
            f"{'Name':<30} {'Grad Mean':>15} {'Grad Std':>15} {'Grad Max':>15} {'Grad Min':>15} {'Grad Norm':>15} {'Weight Norm':>15} {'Ratio':>15} {'Num Zeros':>12}"
        )
        for name, param in model.named_parameters():
            if param.grad is not None:
                if not weight and "weight" in name or not bias and "bias" in name:
                    continue

                grad_mean = param.grad.mean()
                grad_std = param.grad.std()
                grad_max = param.grad.max()
                grad_min = param.grad.min()
                grad_norm = param.grad.norm()
                weight_norm = param.norm()
                ratio = grad_norm / (weight_norm + 1e-6)
                num_zeros = (param.grad == 0).sum().item()

                print(
                    f"{name:<30} {grad_mean:>15.4f} {grad_std:>15.4f} {grad_max:>15.4f} {grad_min:>15.4f} {grad_norm:>15.4f} {weight_norm:>15.4f} {ratio:>15.4f} {num_zeros:>12}"
                )


print_grad_flow(model, csv=True)

In [ ]:
model.eval()
dummy_input = torch.randn(1, 1, cfg.image_size, cfg.image_size, device=DEVICE)

cfg.onnx_path.parent.mkdir(parents=True, exist_ok=True)

torch.onnx.export(
    model,
    dummy_input,
    str(cfg.onnx_path),
    opset_version=cfg.opset_version,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}},
    do_constant_folding=True,
)

# Reload and re-save with all weights inlined (no external .data file).
# onnxruntime-web cannot fetch external data files separately.
onnx_model = onnx.load(str(cfg.onnx_path))
onnx.checker.check_model(onnx_model)
onnx.save_model(onnx_model, str(cfg.onnx_path), save_as_external_data=False)
print(f"ONNX model exported to {cfg.onnx_path} (single-file, no external data)")

In [ ]:
sess_options = ort.SessionOptions()
sess_options.intra_op_num_threads = 1
sess_options.inter_op_num_threads = 1

ort_session = ort.InferenceSession(
    str(cfg.onnx_path), sess_options=sess_options, providers=["CPUExecutionProvider"]
)

test_samples = iter(test_loader)
images, labels = next(test_samples)
test_images = images[:4]
test_labels = labels[:4]

# Run in float32 to match ORT (which always uses float32)
with torch.inference_mode():
    pytorch_outputs = model.float()(test_images.float().to(DEVICE)).cpu().numpy()

ort_inputs = {"input": test_images.numpy()}
ort_outputs = ort_session.run(None, ort_inputs)
ort_logits = ort_outputs[0]

assert np.allclose(pytorch_outputs, ort_logits, atol=1e-5), (
    "Output mismatch between PyTorch and ONNX Runtime!"
)
print("✓ PyTorch and ONNX Runtime outputs match!")
print(f"PyTorch logits shape: {pytorch_outputs.shape}")
print(f"ONNX Runtime logits shape: {ort_logits.shape}")
print(f"Max difference: {np.abs(pytorch_outputs - ort_logits).max():.2e}")


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3))

for i in range(4):
    img = test_images[i].squeeze().numpy()
    true_label = test_labels[i].item()
    pytorch_pred = pytorch_outputs[i].argmax()
    ort_pred = ort_logits[i].argmax()

    axes[i].imshow(img, cmap="gray")
    axes[i].set_title(f"True: {true_label}\nPyTorch: {pytorch_pred}\nORT: {ort_pred}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
HF_REPO_ID = "karanravindra/mnist-cnn-onnx"
HF_MODEL_FILE = "mnist_cnn.onnx"

api = HfApi()

api.create_repo(repo_id=HF_REPO_ID, repo_type="model", private=False, exist_ok=True)
api.upload_file(
    path_or_fileobj=str(cfg.onnx_path),
    path_in_repo=HF_MODEL_FILE,
    repo_id=HF_REPO_ID,
    repo_type="model",
)

model_url = (
    f"https://huggingface.co/{HF_REPO_ID}/resolve/main/{HF_MODEL_FILE}?download=1"
)
print(f"Model uploaded: {model_url}")